### Building the 2019 master dataset

This notebook creates a single master dataset from the monthly NYC yellow taxi trip files for 2019.

The main objective of this step is not yet advanced cleaning or modeling. The purpose here is to:
- locate all monthly raw files,
- standardize their structure,
- sample a fixed number of observations per month,
- combine them into one consistent table,
- and save the result in parquet format for later analysis.

A fixed sample per month is used so that each month contributes equally to the final dataset. This helps control the size of the data while still preserving seasonal variation across the year.

The output of this notebook is therefore a structured base dataset that can be used later for cleaning, feature engineering, and modeling.

In [1]:
from pathlib import Path
import pandas as pd

### Defining paths and general settings

This first block prepares the working environment.

Instead of hardcoding a folder path, the notebook searches upward from the current directory until it finds a folder called `data`. This makes the notebook more robust, especially when it is run from inside a `notebooks/` folder or from another subdirectory.

We also define:
- the input folder, where the raw CSV files are stored,
- the output folder, where the processed parquet file will be saved,
- the sample size per month,
- and a random seed to make the sampling reproducible.

Using a fixed random state is important because it guarantees that the same rows are selected each time the notebook is run.

In [2]:
# Robust project root (works even if the notebook is executed from notebooks/)
cwd = Path().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "data").exists())

path_input = project_root / "data" / "raw"
path_output = project_root / "data" / "procesed"   # kept as originally named
path_output.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_MONTH = 1_000_000
RANDOM_STATE = 42

print("project_root:", project_root)
print("path_input:", path_input)
print("path_output:", path_output)

project_root: C:\Users\leodo\Desktop\NYC-Taxi-ML
path_input: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\raw
path_output: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\procesed


### Finding the monthly files

The raw dataset is organized by month, so the next step is to identify all CSV files corresponding to 2019.

This function searches for files following the naming pattern:

`yellow_tripdata_2019-*.csv`

If no files are found, an error is raised. This is useful because it stops the notebook early and makes the problem visible immediately instead of failing later in a less clear way.

In [3]:
def find_month_files(input_dir: Path, year: int = 2019) -> list[Path]:
    files = sorted(input_dir.glob(f"yellow_tripdata_{year}-*.csv"))
    if not files:
        raise FileNotFoundError(f"No monthly files found in {input_dir} for year={year}")
    return files

files_2019 = find_month_files(path_input, 2019)
[f.name for f in files_2019][:5], len(files_2019)

(['yellow_tripdata_2019-01.csv',
  'yellow_tripdata_2019-02.csv',
  'yellow_tripdata_2019-03.csv',
  'yellow_tripdata_2019-04.csv',
  'yellow_tripdata_2019-05.csv'],
 12)

### Identifying the full set of columns

Monthly files do not always have exactly the same structure. Some months may contain columns that others do not.

To avoid problems when concatenating the files, we first compute the union of all column names present across the 2019 files.

This decision is important because:
- it preserves all available variables,
- it prevents column mismatch errors,
- and it makes the final dataset structurally consistent.

Any missing column in a specific month will later be filled with missing values (`NA`) when that file is aligned to the full column list.

In [4]:
def get_union_columns(files: list[Path]) -> list[str]:
    union = set()
    for f in files:
        cols = pd.read_csv(f, nrows=0).columns.tolist()
        union.update(cols)
    # stable order: keep TLC-like ordering by sorting
    return sorted(union)

all_cols = get_union_columns(files_2019)
print("Total union columns:", len(all_cols))
all_cols

Total union columns: 18


['DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime',
 'trip_distance']

### Reading, sampling, and aligning each monthly file

This is the main preparation step for each monthly file.

For every month, the function does four things:

1. It reads the CSV file into memory.
2. It takes a random sample only if the file has more than the target number of rows.
3. It reindexes the dataset so that its columns match the full union of columns found earlier.
4. It adds metadata variables:
   - `year`
   - `month`
   - `source_file`

These metadata fields are useful for traceability. They make it possible to identify where each observation came from and allow later comparisons by month.

The reindexing step is especially important. Without it, the monthly files could not be combined safely if they differ in structure.

In [5]:
def month_from_filename(file_path: Path) -> int:
    # yellow_tripdata_2019-01.csv -> 01
    return int(file_path.stem.split("_")[-1].split("-")[1])

def read_and_sample_month(file_path: Path, all_cols: list[str],
                          n: int = 1_000_000, random_state: int = 42) -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)

    # sample (only if needed)
    if len(df) > n:
        df = df.sample(n=n, random_state=random_state)

    # align columns to union (adds missing columns as NA)
    df = df.reindex(columns=all_cols)

    # add metadata
    df["year"] = 2019
    df["month"] = month_from_filename(file_path)
    df["source_file"] = file_path.name

    return df

### Building the master table

Now each monthly file is processed using the same logic and stored temporarily in a list.

After all months are prepared, they are concatenated into a single master dataset.

This step creates the first unified version of the 2019 taxi data. At this point:
- the dataset is already standardized across months,
- the monthly contribution is balanced through sampling,
- and the result is ready for more detailed cleaning in the next notebook.

The print statement inside the loop is useful for monitoring progress and checking that each month is being processed correctly.

In [6]:
parts = []
for f in files_2019:
    df_m = read_and_sample_month(f, all_cols, SAMPLE_PER_MONTH, RANDOM_STATE)
    parts.append(df_m)
    print(f"{f.name}: sampled {len(df_m):,} rows | cols={df_m.shape[1]:,}")

master = pd.concat(parts, ignore_index=True)
print("MASTER SHAPE:", master.shape)
master.head()

yellow_tripdata_2019-01.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-02.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-03.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-04.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-05.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-06.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-07.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-08.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-09.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-10.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-11.csv: sampled 1,000,000 rows | cols=21
yellow_tripdata_2019-12.csv: sampled 1,000,000 rows | cols=21
MASTER SHAPE: (12000000, 21)


,DOLocationID,PULocationID,RatecodeID,VendorID,congestion_surcharge,extra,fare_amount,improvement_surcharge,mta_tax,passenger_count,...,store_and_fwd_flag,tip_amount,tolls_amount,total_amount,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance,year,month,source_file
0,234,164,1.0,2.0,NaN,0.0,4.5,0.3,0.5,1.0,...,N,1.06,0.0,6.36,2019-01-09 09:31:43,2019-01-09 09:28:06,0.52,2019,1,yellow_tripdata_2019-01.csv
1,230,100,1.0,2.0,NaN,0.0,7.0,0.3,0.5,1.0,...,N,1.95,0.0,9.75,2019-01-02 07:37:09,2019-01-02 07:29:10,1.15,2019,1,yellow_tripdata_2019-01.csv
2,162,140,1.0,2.0,NaN,1.0,10.5,0.3,0.5,1.0,...,N,0.00,0.0,12.30,2019-01-07 16:06:42,2019-01-07 15:55:27,2.44,2019,1,yellow_tripdata_2019-01.csv
3,239,151,1.0,1.0,NaN,0.0,5.5,0.3,0.5,1.0,...,N,1.25,0.0,7.55,2019-01-09 06:56:05,2019-01-09 06:52:41,1.20,2019,1,yellow_tripdata_2019-01.csv
4,260,140,1.0,1.0,NaN,0.0,20.0,0.3,0.5,1.0,...,N,0.00,0.0,20.80,2019-01-17 09:14:37,2019-01-17 08:50:24,4.60,2019,1,yellow_tripdata_2019-01.csv


### Saving the dataset in parquet format

The final master table is saved as a parquet file.

Parquet is preferred here over CSV because it is:
- more efficient in storage,
- faster to load in later notebooks,
- and better suited for large tabular datasets.

Saving the output at this stage is a practical decision. It prevents having to repeat the whole loading and sampling process every time we want to continue with cleaning or modeling.

In [7]:
out_file = path_output / "master_2019_1M_per_month.parquet"
master.to_parquet(out_file, index=False)
print("Saved:", out_file)

Saved: C:\Users\leodo\Desktop\NYC-Taxi-ML\data\procesed\master_2019_1M_per_month.parquet


### Final note

This notebook does not yet perform analytical cleaning decisions such as removing invalid trips, handling missing values, or engineering variables like trip duration or speed.

Its role is to create a reliable and reproducible base dataset.

In summary, the main decisions made in this notebook are:

- use all monthly files from 2019,
- keep a fixed sample size per month to control dataset size and maintain balance,
- standardize the structure through the union of columns,
- preserve traceability with metadata columns,
- and save the result in parquet format for efficiency.

This makes the next stages of the project cleaner, more consistent, and easier to document.